In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

from datasets import load_dataset
from torchvision.models import resnet18

# Configuration

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 100
LEARNING_RATE = 1e-3
IMG_SIZE = 224
NUM_CLASSES = 2

SAVE_PATH = "/content/drive/MyDrive/resnet18_storyreasoning_100epochs.pt"

# Load dataset from Hugging Face

dataset = load_dataset("daniel3303/StoryReasoning")

def detect_text_column(sample):
    for k, v in sample.items():
        if isinstance(v, str):
            return k
    raise ValueError("No text column found")


def derive_binary_label(item):
    # Example: Assign label 1 if 'chain_of_thought' is not empty, else 0
    # This is an arbitrary placeholder logic. Modify as per your task.
    if 'chain_of_thought' in item and item['chain_of_thought'] and len(item['chain_of_thought']) > 0:
        return 1  # For example, 'chain_of_thought' exists and is not empty
    else:
        return 0  # For example, 'chain_of_thought' is empty or missing

TEXT_COLUMN = "story" # Changed to 'story' as a plausible input for the image encoding
LABEL_COLUMN = "derived_label" # Using a placeholder name as the label is now derived

print(f"Text column selected for image encoding: {TEXT_COLUMN}")
print(f"Label will be derived using 'derive_binary_label' function.")

# Text to Image Encoding
def text_to_image(text, img_size=IMG_SIZE):
    flat_size = img_size * img_size
    buffer = np.zeros(flat_size, dtype=np.uint8)

    encoded = text.encode("utf-8")[:flat_size]
    buffer[:len(encoded)] = list(encoded)

    img = buffer.reshape(img_size, img_size)
    img = np.stack([img, img, img])

    return torch.tensor(img, dtype=torch.float32) / 255.0

class StoryReasoningImageDataset(Dataset):
    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = text_to_image(item[TEXT_COLUMN])
        # Use the derived label
        label = derive_binary_label(item)
        return image, label

train_dataset = StoryReasoningImageDataset(dataset["train"])
val_dataset = StoryReasoningImageDataset(dataset["test"]) # Changed from "validation" to "test"

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Model

model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model.to(DEVICE)

# Optimizer & Loss
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training Loop (100 epochs)
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total if total > 0 else 0

    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch [{epoch+1:03}/{EPOCHS}] | "
            f"Train Loss: {avg_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )
# Save to Google Drive
torch.save(model.state_dict(), SAVE_PATH)
print(f"✅ Model saved to Google Drive at:\n{SAVE_PATH}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/327M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/331M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/115M [00:00<?, ?B/s]

...

Text column selected for image encoding: story
Label will be derived using 'derive_binary_label' function.


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 193MB/s]


Epoch [005/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [010/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [015/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [020/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [025/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [030/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [035/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [040/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [045/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [050/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [055/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [060/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [065/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [070/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [075/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [080/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [085/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [090/100] | Train Loss: 0.0000 | Val Acc: 1.0000
Epoch [095

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['story_id', 'images', 'frame_count', 'chain_of_thought', 'story'],
        num_rows: 3552
    })
    test: Dataset({
        features: ['story_id', 'images', 'frame_count', 'chain_of_thought', 'story'],
        num_rows: 626
    })
})
